<a href="https://colab.research.google.com/github/nesbita/ML-Collision-Severity-YOLO-Detection/blob/main/Ariana_Nesbit_Ass1_T11.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# DATA PREPARATION


from google.colab import drive
import pandas as pd
import numpy as np
import pickle
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, OneHotEncoder

In [ ]:
drive.mount('/content/drive')

DATA_PATH = '/content/drive/MyDrive/Colab Notebooks/dft-road-casualty-statistics-collision-2024.csv'

# display all columns and 100 rows at a time
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

df = pd.read_csv(DATA_PATH)

print(df.info())
print(f'Columns: {df.columns.tolist()}')


Mounted at /content/drive
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100927 entries, 0 to 100926
Data columns (total 44 columns):
 #   Column                                            Non-Null Count   Dtype  
---  ------                                            --------------   -----  
 0   collision_index                                   100927 non-null  object 
 1   collision_year                                    100927 non-null  int64  
 2   collision_ref_no                                  100927 non-null  object 
 3   location_easting_osgr                             100927 non-null  int64  
 4   location_northing_osgr                            100927 non-null  int64  
 5   longitude                                         100927 non-null  float64
 6   latitude                                          100927 non-null  float64
 7   police_force                                      100927 non-null  int64  
 8   collision_severity                                100927 n

/tmp/ipykernel_12456/2173920014.py:9: DtypeWarning: Columns (0,2) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(DATA_PATH)


In [ ]:
## LEAKAGE AWARENESS


# given removals (collision identifiers, district and road IDs)
GIVEN_REMOVALS = [
    'collision_index', 'collision_year', 'collision_ref_no',
    'location_easting_osgr', 'location_northing_osgr',
    'police_force', 'local_authority_district',
    'local_authority_ons_district', 'local_authority_highway',
    'local_authority_highway_current', 'first_road_number',
    'junction_detail_historic', 'pedestrian_crossing_human_control_historic',
    'pedestrian_crossing_physical_facilities_historic',
    'carriageway_hazards_historic', 'second_road_number',
    'did_police_officer_attend_scene_of_accident', 'lsoa_of_accident_location',
]
# additional features that will cause leakage
ADDITIONAL_REMOVALS = [
    'enhanced_severity_collision', 'collision_injury_based',
    'collision_adjusted_severity_serious', 'collision_adjusted_severity_slight',
    'number_of_casualties', 'number_of_vehicles'
]

REMOVE_COLUMNS = GIVEN_REMOVALS + ADDITIONAL_REMOVALS

df_clean = df.drop(columns=REMOVE_COLUMNS)
print(df_clean.shape)
print(df_clean.info())


(100927, 20)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100927 entries, 0 to 100926
Data columns (total 20 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   longitude                   100927 non-null  float64
 1   latitude                    100927 non-null  float64
 2   collision_severity          100927 non-null  int64  
 3   date                        100927 non-null  object 
 4   day_of_week                 100927 non-null  int64  
 5   time                        100927 non-null  object 
 6   first_road_class            100927 non-null  int64  
 7   road_type                   100927 non-null  int64  
 8   speed_limit                 100927 non-null  int64  
 9   junction_detail             100927 non-null  int64  
 10  junction_control            100927 non-null  int64  
 11  second_road_class           100927 non-null  int64  
 12  pedestrian_crossing         100927 non-null  int64  
 13  l

In [ ]:
# missing/unknown values
# -1 = data missing or out of range
#  9 = unknown
# 99 = unknown (self reported)

MISSING_AND_UNKNOWN_CODES = {
    'junction_detail':          [-1, 99],
    'junction_control':         [-1, 9],
    'second_road_class':        [-1],
    'pedestrian_crossing':      [-1, 99],
    'light_conditions':         [-1],
    'weather_conditions':       [9],
    'road_surface_conditions':  [-1, 9],
    'special_conditions_at_site': [-1, 9],
    'carriageway_hazards':      [-1, 99],
    'trunk_road_flag':          [-1],
    'speed_limit':              [-1],
}

# go through each of the columns and codes in the dictionary and
# if the codes are -1, 9, or 99, replace them with not a number

for col, codes in MISSING_AND_UNKNOWN_CODES.items():
  if col in df_clean.columns and codes:
    df_clean[col] = df_clean[col].replace(codes, np.nan)

print(df_clean.isnull().sum())
print(df_clean.info())

df_clean = df_clean.drop(columns='special_conditions_at_site')
print(df_clean.info())


longitude                         0
latitude                          0
collision_severity                0
date                              0
day_of_week                       0
time                              0
first_road_class                  0
road_type                         0
speed_limit                       3
junction_detail                7007
junction_control              44489
second_road_class             11167
pedestrian_crossing            4046
light_conditions                  6
weather_conditions             3145
road_surface_conditions        2374
special_conditions_at_site    61971
carriageway_hazards            3083
urban_or_rural_area               0
trunk_road_flag                7080
dtype: int64
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100927 entries, 0 to 100926
Data columns (total 20 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   longitude                   100927 

In [ ]:
# format date, month, and hour
df_clean['date'] = pd.to_datetime(df_clean['date'], format='mixed')
df_clean['month'] = df_clean['date'].dt.month
df_clean['time'] = pd.to_datetime(df_clean['time'], format='%H:%M', errors='coerce')
df_clean['hour'] = df_clean['time'].dt.hour
print(df_clean.info())

# drop original date and time
df_clean = df_clean.drop(columns=['date', 'time'])
print(df_clean.info())

# add is_weekend
#df_clean['is_weekend'] = df_clean['day_of_week'].isin([1, 7]).astype(int)

# is_rush_hour

# is_dark


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100927 entries, 0 to 100926
Data columns (total 21 columns):
 #   Column                   Non-Null Count   Dtype         
---  ------                   --------------   -----         
 0   longitude                100927 non-null  float64       
 1   latitude                 100927 non-null  float64       
 2   collision_severity       100927 non-null  int64         
 3   date                     100927 non-null  datetime64[ns]
 4   day_of_week              100927 non-null  int64         
 5   time                     100927 non-null  datetime64[ns]
 6   first_road_class         100927 non-null  int64         
 7   road_type                100927 non-null  int64         
 8   speed_limit              100924 non-null  float64       
 9   junction_detail          93920 non-null   float64       
 10  junction_control         56438 non-null   float64       
 11  second_road_class        89760 non-null   float64       
 12  pedestrian_cross

In [ ]:
## PREPROCESSING DESIGN

NUMERIC_COLUMNS = ['longitude', 'latitude', 'speed_limit', 'hour', 'month']

CATEGORICAL_COLUMNS = ['day_of_week', 'first_road_class', 'road_type',
                    'junction_detail', 'junction_control', 'second_road_class',
                    'pedestrian_crossing', 'light_conditions', 'weather_conditions',
                    'road_surface_conditions', 'carriageway_hazards', 'trunk_road_flag']

BINARY_COLUMNS = ['urban_or_rural_area']

# numeric: median imputation + scaling
numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')), #median
    ('scaler', StandardScaler()),
])

# categorical: mode imputation + ordinal encoding (for tree models)
categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')), #mode
    ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)),
])

# binary
binary_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')), #mode
])

# combine into one preprocessor
preprocessor = ColumnTransformer([
    ('num', numeric_pipeline, NUMERIC_COLUMNS),
    ('cat', categorical_pipeline, CATEGORICAL_COLUMNS),
    ('bin', binary_pipeline, BINARY_COLUMNS),
])

print('Preprocessing pipeline defined.')

Preprocessing pipeline defined.


In [ ]:
## SAVE

# save as  without first column
SAVE_PATH = '/content/drive/MyDrive/Colab Notebooks/'
df_clean.to_csv(SAVE_PATH + 'df_clean.csv', index=False)

# save columns
feature_config = {
    'NUMERIC_COLUMNS': NUMERIC_COLUMNS,
    'CATEGORICAL_COLUMNS': CATEGORICAL_COLUMNS,
    'BINARY_COLUMNS': BINARY_COLUMNS,
}

# for opening in T1.2
with open(SAVE_PATH + 'feature_config.pkl', 'wb') as f:
    pickle.dump(feature_config, f)

# save preprocessor
with open(SAVE_PATH + 'preprocessor.pkl', 'wb') as f:
    pickle.dump(preprocessor, f)

print('Saved successfully.')

Saved successfully.
